# Research PDF Brain - Google Drive Integration

A comprehensive system for organizing, analyzing, and searching through research PDFs stored in Google Drive.

## Features:
- PDF ingestion from Google Drive
- Intelligent parsing and chunking
- Metadata extraction and GPT-based summarization
- Three-tier topic taxonomy
- Automated classification
- Vector embeddings with FAISS indexing
- RAG-powered literature search
- LangGraph agentic workflow

## Installation and Setup

In [ ]:
# Install required packages
!pip install -q langchain langchain-openai langchain-community langgraph
!pip install -q pypdf2 pymupdf pdfplumber
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client
!pip install -q pandas numpy scikit-learn
!pip install -q tiktoken
!pip install -q python-dotenv

In [ ]:
# Import required libraries
import os
import io
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional, TypedDict, Annotated
from datetime import datetime
import pickle

# Google Drive
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# PDF Processing
import PyPDF2
import fitz  # PyMuPDF
import pdfplumber

# LangChain and LangGraph
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

# FAISS
import faiss

# Other utilities
from sentence_transformers import SentenceTransformer
import tiktoken

## Google Drive Authentication and Setup

In [ ]:
# Authenticate with Google Drive
auth.authenticate_user()
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuration
class Config:
    """Configuration for the Research PDF Brain"""
    
    # Google Drive settings
    DRIVE_FOLDER_PATH = '/content/drive/MyDrive/Research_PDFs'  # Change this to your folder
    OUTPUT_FOLDER = '/content/drive/MyDrive/Research_Brain_Output'
    
    # OpenAI API settings
    OPENAI_API_KEY = ''  # Set your OpenAI API key
    MODEL_NAME = 'gpt-4-turbo-preview'  # Using GPT-4 (GPT-5.1 not yet available)
    EMBEDDING_MODEL = 'text-embedding-3-large'
    
    # Processing settings
    CHUNK_SIZE = 1000
    CHUNK_OVERLAP = 200
    MAX_TOKENS_PER_CHUNK = 500
    
    # Taxonomy levels
    TIER_1_CATEGORIES = ['Computer Science', 'Physics', 'Mathematics', 'Biology', 'Chemistry', 'Engineering', 'Other']
    
    # FAISS settings
    EMBEDDING_DIMENSION = 3072  # For text-embedding-3-large
    
config = Config()

# Set OpenAI API key
if config.OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = config.OPENAI_API_KEY
else:
    print("⚠️ Warning: Please set your OpenAI API key in the Config class")

In [ ]:
# Create output directory
os.makedirs(config.OUTPUT_FOLDER, exist_ok=True)

## LangGraph State Definition

In [ ]:
class PaperState(TypedDict):
    """State for processing a single research paper"""
    file_path: str
    file_name: str
    raw_text: str
    chunks: List[str]
    metadata: Dict
    summary: str
    tier1_category: str
    tier2_category: str
    tier3_category: str
    embeddings: Optional[np.ndarray]
    error: Optional[str]

class CorpusState(TypedDict):
    """State for the entire corpus"""
    papers: List[PaperState]
    master_table: Optional[pd.DataFrame]
    faiss_index: Optional[Any]
    taxonomy: Dict
    total_papers: int
    processed_papers: int
    errors: List[str]

## PDF Ingestion and Parsing

In [ ]:
def get_pdf_files(folder_path: str) -> List[str]:
    """Get all PDF files from the specified folder"""
    pdf_files = []
    
    if os.path.exists(folder_path):
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if file.lower().endswith('.pdf'):
                    pdf_files.append(os.path.join(root, file))
    
    return pdf_files

def extract_text_from_pdf(pdf_path: str) -> Tuple[str, Dict]:
    """Extract text and metadata from PDF using multiple methods for robustness"""
    text = ""
    metadata = {}
    
    try:
        # Try PyMuPDF first (most reliable)
        doc = fitz.open(pdf_path)
        
        # Extract metadata
        metadata = {
            'title': doc.metadata.get('title', os.path.basename(pdf_path)),
            'author': doc.metadata.get('author', 'Unknown'),
            'subject': doc.metadata.get('subject', ''),
            'keywords': doc.metadata.get('keywords', ''),
            'creator': doc.metadata.get('creator', ''),
            'producer': doc.metadata.get('producer', ''),
            'creation_date': doc.metadata.get('creationDate', ''),
            'page_count': len(doc)
        }
        
        # Extract text
        for page in doc:
            text += page.get_text()
        
        doc.close()
        
    except Exception as e:
        print(f"Error with PyMuPDF for {pdf_path}: {e}")
        
        # Fallback to PyPDF2
        try:
            with open(pdf_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                metadata['page_count'] = len(reader.pages)
                metadata['title'] = os.path.basename(pdf_path)
                
                for page in reader.pages:
                    text += page.extract_text()
        except Exception as e2:
            print(f"Error with PyPDF2 for {pdf_path}: {e2}")
            text = ""
    
    return text, metadata

## Text Chunking

In [ ]:
def chunk_text(text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> List[str]:
    """Split text into overlapping chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    chunks = text_splitter.split_text(text)
    return chunks

## GPT-Based Summarization and Classification

In [ ]:
class PaperAnalyzer:
    """Handles GPT-based analysis of research papers"""
    
    def __init__(self, model_name: str = 'gpt-4-turbo-preview'):
        self.llm = ChatOpenAI(model=model_name, temperature=0.3)
    
    def generate_summary(self, text: str, metadata: Dict) -> str:
        """Generate a comprehensive summary of the paper"""
        # Truncate text if too long
        max_chars = 15000
        truncated_text = text[:max_chars] if len(text) > max_chars else text
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert research assistant specializing in academic literature analysis."),
            ("user", """Analyze the following research paper and provide a comprehensive summary.

Paper Title: {title}
Author(s): {author}

Text (truncated if necessary):
{text}

Please provide:
1. A concise abstract (2-3 sentences)
2. Main contributions
3. Key methodologies
4. Primary findings
5. Significance and potential impact

Format the response as a structured summary.""")
        ])
        
        messages = prompt.format_messages(
            title=metadata.get('title', 'Unknown'),
            author=metadata.get('author', 'Unknown'),
            text=truncated_text
        )
        
        response = self.llm.invoke(messages)
        return response.content
    
    def classify_paper(self, text: str, summary: str, metadata: Dict) -> tuple[str, str, str]:
        """Classify paper into three-tier taxonomy"""
        # Truncate text if too long
        max_chars = 10000
        truncated_text = text[:max_chars] if len(text) > max_chars else text
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert at classifying research papers into a hierarchical taxonomy."),
            ("user", """Classify the following research paper into a three-tier taxonomy.

Paper Title: {title}
Summary: {summary}
Text Sample: {text}

Please classify into:
1. Tier 1 (Broad Field): Choose from [Computer Science, Physics, Mathematics, Biology, Chemistry, Engineering, Other]
2. Tier 2 (Sub-field): A specific subfield within the Tier 1 category (e.g., Machine Learning, Quantum Physics, etc.)
3. Tier 3 (Specific Topic): The most specific topic or methodology (e.g., Transformer Models, Quantum Entanglement, etc.)

Respond in this exact format:
TIER1: [category]
TIER2: [subcategory]
TIER3: [specific topic]""")
        ])
        
        messages = prompt.format_messages(
            title=metadata.get('title', 'Unknown'),
            summary=summary[:500],  # Use first 500 chars of summary
            text=truncated_text
        )
        
        response = self.llm.invoke(messages)
        content = response.content
        
        # Parse the response
        tier1 = tier2 = tier3 = "Unknown"
        
        for line in content.split('\n'):
            if line.startswith('TIER1:'):
                tier1 = line.replace('TIER1:', '').strip()
            elif line.startswith('TIER2:'):
                tier2 = line.replace('TIER2:', '').strip()
            elif line.startswith('TIER3:'):
                tier3 = line.replace('TIER3:', '').strip()
        
        return tier1, tier2, tier3

analyzer = PaperAnalyzer(config.MODEL_NAME)

## Embedding Generation

In [ ]:
class EmbeddingGenerator:
    """Handles generation of embeddings for text"""
    
    def __init__(self, model_name: str = 'text-embedding-3-large'):
        self.embeddings = OpenAIEmbeddings(model=model_name)
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        if not texts:
            return np.array([])
        
        # Generate embeddings
        embedded = self.embeddings.embed_documents(texts)
        return np.array(embedded)
    
    def generate_embedding(self, text: str) -> np.ndarray:
        """Generate embedding for a single text"""
        embedded = self.embeddings.embed_query(text)
        return np.array(embedded)

embedding_generator = EmbeddingGenerator(config.EMBEDDING_MODEL)

## LangGraph Workflow Definition

In [ ]:
def ingest_paper(state: PaperState) -> PaperState:
    """Node: Ingest and parse PDF"""
    print(f"📄 Ingesting: {state['file_name']}")
    
    try:
        text, metadata = extract_text_from_pdf(state['file_path'])
        state['raw_text'] = text
        state['metadata'] = metadata
        
        if not text or len(text) < 100:
            state['error'] = "Failed to extract meaningful text from PDF"
    except Exception as e:
        state['error'] = f"Ingestion error: {str(e)}"
    
    return state

def chunk_paper(state: PaperState) -> PaperState:
    """Node: Chunk the paper text"""
    if state.get('error'):
        return state
    
    print(f"✂️  Chunking: {state['file_name']}")
    
    try:
        chunks = chunk_text(state['raw_text'], config.CHUNK_SIZE, config.CHUNK_OVERLAP)
        state['chunks'] = chunks
    except Exception as e:
        state['error'] = f"Chunking error: {str(e)}"
    
    return state

def summarize_paper(state: PaperState) -> PaperState:
    """Node: Generate summary using GPT"""
    if state.get('error'):
        return state
    
    print(f"📝 Summarizing: {state['file_name']}")
    
    try:
        summary = analyzer.generate_summary(state['raw_text'], state['metadata'])
        state['summary'] = summary
    except Exception as e:
        state['error'] = f"Summarization error: {str(e)}"
        state['summary'] = "Summary generation failed"
    
    return state

def classify_paper(state: PaperState) -> PaperState:
    """Node: Classify paper into taxonomy"""
    if state.get('error'):
        return state
    
    print(f"🏷️  Classifying: {state['file_name']}")
    
    try:
        tier1, tier2, tier3 = analyzer.classify_paper(
            state['raw_text'], 
            state['summary'], 
            state['metadata']
        )
        state['tier1_category'] = tier1
        state['tier2_category'] = tier2
        state['tier3_category'] = tier3
    except Exception as e:
        state['error'] = f"Classification error: {str(e)}"
        state['tier1_category'] = state['tier2_category'] = state['tier3_category'] = "Unknown"
    
    return state

def generate_embeddings_node(state: PaperState) -> PaperState:
    """Node: Generate embeddings for paper chunks"""
    if state.get('error'):
        return state
    
    print(f"🔢 Generating embeddings: {state['file_name']}")
    
    try:
        # Generate embeddings for chunks
        if state['chunks']:
            embeddings = embedding_generator.generate_embeddings(state['chunks'])
            state['embeddings'] = embeddings
    except Exception as e:
        state['error'] = f"Embedding error: {str(e)}"
    
    return state

# Create the workflow graph
def create_paper_processing_graph():
    """Create LangGraph workflow for processing a single paper"""
    workflow = StateGraph(PaperState)
    
    # Add nodes
    workflow.add_node("ingest", ingest_paper)
    workflow.add_node("chunk", chunk_paper)
    workflow.add_node("summarize", summarize_paper)
    workflow.add_node("classify", classify_paper)
    workflow.add_node("embed", generate_embeddings_node)
    
    # Define edges
    workflow.set_entry_point("ingest")
    workflow.add_edge("ingest", "chunk")
    workflow.add_edge("chunk", "summarize")
    workflow.add_edge("summarize", "classify")
    workflow.add_edge("classify", "embed")
    workflow.add_edge("embed", END)
    
    return workflow.compile()

paper_graph = create_paper_processing_graph()

## Process All Papers

In [ ]:
def process_corpus(folder_path: str) -> CorpusState:
    """Process all PDFs in the corpus"""
    print("🚀 Starting Research PDF Brain...\n")
    
    # Get all PDF files
    pdf_files = get_pdf_files(folder_path)
    print(f"Found {len(pdf_files)} PDF files\n")
    
    if not pdf_files:
        print("⚠️ No PDF files found in the specified folder.")
        return None
    
    # Initialize corpus state
    corpus_state = {
        'papers': [],
        'master_table': None,
        'faiss_index': None,
        'taxonomy': {},
        'total_papers': len(pdf_files),
        'processed_papers': 0,
        'errors': []
    }
    
    # Process each paper
    for i, pdf_path in enumerate(pdf_files, 1):
        print(f"\n{'='*80}")
        print(f"Processing paper {i}/{len(pdf_files)}")
        print(f"{'='*80}\n")
        
        # Initialize paper state
        paper_state = {
            'file_path': pdf_path,
            'file_name': os.path.basename(pdf_path),
            'raw_text': '',
            'chunks': [],
            'metadata': {},
            'summary': '',
            'tier1_category': '',
            'tier2_category': '',
            'tier3_category': '',
            'embeddings': None,
            'error': None
        }
        
        # Run through the LangGraph workflow
        try:
            result = paper_graph.invoke(paper_state)
            corpus_state['papers'].append(result)
            
            if result.get('error'):
                corpus_state['errors'].append(f"{result['file_name']}: {result['error']}")
            else:
                corpus_state['processed_papers'] += 1
                
        except Exception as e:
            error_msg = f"{os.path.basename(pdf_path)}: {str(e)}"
            corpus_state['errors'].append(error_msg)
            print(f"❌ Error: {error_msg}")
    
    print(f"\n\n{'='*80}")
    print(f"✅ Processing Complete!")
    print(f"{'='*80}")
    print(f"Total papers: {corpus_state['total_papers']}")
    print(f"Successfully processed: {corpus_state['processed_papers']}")
    print(f"Errors: {len(corpus_state['errors'])}")
    
    return corpus_state

## Build FAISS Index

In [ ]:
def build_faiss_index(corpus_state: CorpusState) -> FAISS:
    """Build FAISS index from all paper embeddings"""
    print("\n🔍 Building FAISS index...")
    
    documents = []
    
    for paper in corpus_state['papers']:
        if paper.get('error') or not paper.get('chunks'):
            continue
        
        # Create documents for each chunk
        for i, chunk in enumerate(paper['chunks']):
            metadata = {
                'source': paper['file_name'],
                'chunk_id': i,
                'title': paper['metadata'].get('title', paper['file_name']),
                'author': paper['metadata'].get('author', 'Unknown'),
                'tier1': paper.get('tier1_category', 'Unknown'),
                'tier2': paper.get('tier2_category', 'Unknown'),
                'tier3': paper.get('tier3_category', 'Unknown'),
            }
            doc = Document(page_content=chunk, metadata=metadata)
            documents.append(doc)
    
    print(f"Creating FAISS index with {len(documents)} document chunks...")
    
    # Create FAISS index
    vectorstore = FAISS.from_documents(documents, embedding_generator.embeddings)
    
    # Save the index
    faiss_path = os.path.join(config.OUTPUT_FOLDER, 'faiss_index')
    vectorstore.save_local(faiss_path)
    print(f"✅ FAISS index saved to {faiss_path}")
    
    return vectorstore

## Create Master Table

In [ ]:
def create_master_table(corpus_state: CorpusState) -> pd.DataFrame:
    """Create master table with all paper information"""
    print("\n📊 Creating master table...")
    
    records = []
    
    for paper in corpus_state['papers']:
        record = {
            'file_name': paper['file_name'],
            'file_path': paper['file_path'],
            'title': paper['metadata'].get('title', 'Unknown'),
            'author': paper['metadata'].get('author', 'Unknown'),
            'page_count': paper['metadata'].get('page_count', 0),
            'tier1_category': paper.get('tier1_category', 'Unknown'),
            'tier2_category': paper.get('tier2_category', 'Unknown'),
            'tier3_category': paper.get('tier3_category', 'Unknown'),
            'summary': paper.get('summary', ''),
            'num_chunks': len(paper.get('chunks', [])),
            'has_error': paper.get('error') is not None,
            'error_message': paper.get('error', '')
        }
        records.append(record)
    
    df = pd.DataFrame(records)
    
    # Save to CSV
    csv_path = os.path.join(config.OUTPUT_FOLDER, 'master_table.csv')
    df.to_csv(csv_path, index=False)
    print(f"✅ Master table saved to {csv_path}")
    
    # Save to Excel for better readability
    excel_path = os.path.join(config.OUTPUT_FOLDER, 'master_table.xlsx')
    df.to_excel(excel_path, index=False)
    print(f"✅ Master table saved to {excel_path}")
    
    return df

## Build Taxonomy

In [ ]:
def build_taxonomy(corpus_state: CorpusState) -> Dict:
    """Build hierarchical taxonomy from classified papers"""
    print("\n🌳 Building taxonomy...")
    
    taxonomy = {}
    
    for paper in corpus_state['papers']:
        if paper.get('error'):
            continue
        
        tier1 = paper.get('tier1_category', 'Unknown')
        tier2 = paper.get('tier2_category', 'Unknown')
        tier3 = paper.get('tier3_category', 'Unknown')
        
        # Build hierarchical structure
        if tier1 not in taxonomy:
            taxonomy[tier1] = {}
        
        if tier2 not in taxonomy[tier1]:
            taxonomy[tier1][tier2] = {}
        
        if tier3 not in taxonomy[tier1][tier2]:
            taxonomy[tier1][tier2][tier3] = []
        
        taxonomy[tier1][tier2][tier3].append(paper['file_name'])
    
    # Save taxonomy
    taxonomy_path = os.path.join(config.OUTPUT_FOLDER, 'taxonomy.json')
    with open(taxonomy_path, 'w') as f:
        json.dump(taxonomy, f, indent=2)
    print(f"✅ Taxonomy saved to {taxonomy_path}")
    
    # Print taxonomy summary
    print("\nTaxonomy Summary:")
    for tier1, tier2_dict in taxonomy.items():
        print(f"\n  📁 {tier1}:")
        for tier2, tier3_dict in tier2_dict.items():
            print(f"    📂 {tier2}: {len(tier3_dict)} topics")
    
    return taxonomy

## RAG Search Tools

In [ ]:
class RAGSearchEngine:
    """RAG-powered search engine for research papers"""
    
    def __init__(self, vectorstore: FAISS, llm: ChatOpenAI):
        self.vectorstore = vectorstore
        self.llm = llm
    
    def search(self, query: str, k: int = 5) -> List[Document]:
        """Perform similarity search"""
        results = self.vectorstore.similarity_search(query, k=k)
        return results
    
    def search_with_score(self, query: str, k: int = 5) -> List[tuple]:
        """Perform similarity search with relevance scores"""
        results = self.vectorstore.similarity_search_with_score(query, k=k)
        return results
    
    def answer_question(self, question: str, k: int = 5) -> str:
        """Answer a question using RAG"""
        # Retrieve relevant documents
        docs = self.search(question, k=k)
        
        # Combine context
        context = "\n\n".join([f"Source: {doc.metadata['source']}\n{doc.page_content}" for doc in docs])
        
        # Generate answer
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a research assistant helping to answer questions about academic papers. Use the provided context to answer questions accurately and cite your sources."),
            ("user", """Context from research papers:
{context}

Question: {question}

Please provide a comprehensive answer based on the context above. Cite the specific papers you reference.""")
        ])
        
        messages = prompt.format_messages(context=context, question=question)
        response = self.llm.invoke(messages)
        
        return response.content
    
    def find_similar_papers(self, paper_title: str, k: int = 5) -> List[Document]:
        """Find papers similar to a given paper"""
        return self.search(paper_title, k=k)
    
    def search_by_category(self, tier1: str = None, tier2: str = None, tier3: str = None) -> List[Document]:
        """Search papers by taxonomy category"""
        # This would require filtering - simplified version using text search
        query_parts = []
        if tier1:
            query_parts.append(tier1)
        if tier2:
            query_parts.append(tier2)
        if tier3:
            query_parts.append(tier3)
        
        query = " ".join(query_parts)
        return self.search(query, k=10)

## Main Execution Pipeline

In [ ]:
# Main execution
def run_research_brain():
    """Main function to run the entire pipeline"""
    
    print("""\n
    ╔═══════════════════════════════════════════════════════════════╗
    ║                                                               ║
    ║          🧠 RESEARCH PDF BRAIN - Google Drive Edition          ║
    ║                                                               ║
    ║  Intelligent Literature Organization & RAG-Powered Search     ║
    ║                                                               ║
    ╚═══════════════════════════════════════════════════════════════╝
    \n""")
    
    # Check API key
    if not os.environ.get('OPENAI_API_KEY'):
        print("❌ ERROR: OpenAI API key not set!")
        print("Please set config.OPENAI_API_KEY in the configuration cell above.")
        return None
    
    # Process corpus
    corpus_state = process_corpus(config.DRIVE_FOLDER_PATH)
    
    if corpus_state is None or corpus_state['processed_papers'] == 0:
        print("\n❌ No papers were successfully processed.")
        return None
    
    # Build master table
    master_table = create_master_table(corpus_state)
    corpus_state['master_table'] = master_table
    
    # Build taxonomy
    taxonomy = build_taxonomy(corpus_state)
    corpus_state['taxonomy'] = taxonomy
    
    # Build FAISS index
    vectorstore = build_faiss_index(corpus_state)
    corpus_state['faiss_index'] = vectorstore
    
    # Create RAG search engine
    rag_engine = RAGSearchEngine(vectorstore, analyzer.llm)
    
    # Save corpus state
    state_path = os.path.join(config.OUTPUT_FOLDER, 'corpus_state.pkl')
    with open(state_path, 'wb') as f:
        # Remove non-serializable items before saving
        save_state = corpus_state.copy()
        save_state['faiss_index'] = None  # Already saved separately
        pickle.dump(save_state, f)
    print(f"\n✅ Corpus state saved to {state_path}")
    
    print("\n\n" + "="*80)
    print("🎉 RESEARCH PDF BRAIN INITIALIZATION COMPLETE!")
    print("="*80)
    print(f"\n📊 Results Summary:")
    print(f"   • Total papers processed: {corpus_state['processed_papers']}/{corpus_state['total_papers']}")
    print(f"   • Taxonomy categories (Tier 1): {len(taxonomy)}")
    print(f"   • Output folder: {config.OUTPUT_FOLDER}")
    print(f"\n📁 Generated Files:")
    print(f"   • master_table.csv - Complete paper database")
    print(f"   • master_table.xlsx - Excel format")
    print(f"   • taxonomy.json - Hierarchical topic taxonomy")
    print(f"   • faiss_index/ - Vector database for semantic search")
    print(f"   • corpus_state.pkl - Complete processing state")
    
    return corpus_state, rag_engine

# Run the pipeline
corpus_state, rag_engine = run_research_brain()

## Interactive Search Interface

In [ ]:
# Example: Search for papers
def search_papers(query: str, num_results: int = 5):
    """Search for papers using semantic search"""
    print(f"\n🔍 Searching for: '{query}'\n")
    print("="*80)
    
    results = rag_engine.search_with_score(query, k=num_results)
    
    for i, (doc, score) in enumerate(results, 1):
        print(f"\n{i}. {doc.metadata['title']}")
        print(f"   Author: {doc.metadata['author']}")
        print(f"   Category: {doc.metadata['tier1']} > {doc.metadata['tier2']} > {doc.metadata['tier3']}")
        print(f"   Relevance Score: {score:.4f}")
        print(f"   Excerpt: {doc.page_content[:200]}...")
        print("   " + "-"*76)

# Example usage:
# search_papers("machine learning transformers", num_results=3)

In [ ]:
# Example: Ask a question about the research
def ask_question(question: str):
    """Ask a question and get an answer based on the papers"""
    print(f"\n❓ Question: {question}\n")
    print("="*80)
    
    answer = rag_engine.answer_question(question, k=5)
    
    print(f"\n💡 Answer:\n")
    print(answer)
    print("\n" + "="*80)

# Example usage:
# ask_question("What are the main approaches to neural network optimization?")

In [ ]:
# Example: View taxonomy statistics
def show_taxonomy_stats():
    """Display taxonomy statistics"""
    if corpus_state and 'master_table' in corpus_state:
        df = corpus_state['master_table']
        
        print("\n📊 Taxonomy Statistics\n")
        print("="*80)
        
        print("\nTier 1 Distribution:")
        print(df['tier1_category'].value_counts())
        
        print("\n\nTier 2 Distribution (Top 10):")
        print(df['tier2_category'].value_counts().head(10))
        
        print("\n\nTier 3 Distribution (Top 10):")
        print(df['tier3_category'].value_counts().head(10))

# Example usage:
# show_taxonomy_stats()

## Load Previously Processed Data

In [ ]:
def load_existing_corpus():
    """Load previously processed corpus data"""
    print("📂 Loading existing corpus data...\n")
    
    # Load corpus state
    state_path = os.path.join(config.OUTPUT_FOLDER, 'corpus_state.pkl')
    if not os.path.exists(state_path):
        print("❌ No existing corpus state found. Please run the pipeline first.")
        return None, None
    
    with open(state_path, 'rb') as f:
        corpus_state = pickle.load(f)
    
    # Load FAISS index
    faiss_path = os.path.join(config.OUTPUT_FOLDER, 'faiss_index')
    if not os.path.exists(faiss_path):
        print("❌ No FAISS index found. Please run the pipeline first.")
        return None, None
    
    vectorstore = FAISS.load_local(faiss_path, embedding_generator.embeddings, allow_dangerous_deserialization=True)
    corpus_state['faiss_index'] = vectorstore
    
    # Create RAG engine
    rag_engine = RAGSearchEngine(vectorstore, analyzer.llm)
    
    print("✅ Corpus data loaded successfully!")
    print(f"   • Papers: {corpus_state['processed_papers']}")
    print(f"   • Taxonomy categories: {len(corpus_state.get('taxonomy', {}))}")
    
    return corpus_state, rag_engine

# Example usage:
# corpus_state, rag_engine = load_existing_corpus()

## Usage Examples

After running the main pipeline, you can use these functions:

```python
# 1. Search for papers on a specific topic
search_papers("deep learning optimization", num_results=5)

# 2. Ask questions about your research corpus
ask_question("What are the latest advances in transformer architectures?")

# 3. View taxonomy statistics
show_taxonomy_stats()

# 4. Access the master table
corpus_state['master_table'].head()

# 5. Explore the taxonomy
print(json.dumps(corpus_state['taxonomy'], indent=2))

# 6. Find similar papers
similar = rag_engine.find_similar_papers("Attention Is All You Need", k=5)
for doc in similar:
    print(doc.metadata['title'])
```